# B7. Review Intelligent Analysis Notebook

> **Related Module**: [B7 Review NLP System](../paths/b-developers/b7-review-nlp-system.md)
>
> **Features**: Sentiment analysis + BERTopic topic modeling + LLM insight generation
>
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kangise/ecommerce-ai-roadmap/blob/main/notebooks/b7-review-analysis.ipynb)

---

## 1. Install Dependencies

In [ ]:
!pip install -q pandas numpy transformers sentence-transformers bertopic vaderSentiment wordcloud plotly matplotlib

## 2. Load Data

Upload your Review CSV file, or use the sample data.
The CSV should contain: `rating`, `title`, `body`, `date` columns.

In [ ]:
import pandas as pd
import numpy as np
import re

# === Option 1: Upload your CSV ===
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_csv(list(uploaded.keys())[0])

# === Option 2: Use sample data ===
np.random.seed(42)
n = 500
ratings = np.random.choice([1,2,3,4,5], n, p=[0.05, 0.08, 0.12, 0.30, 0.45])
titles = [
    'Great product, love it!', 'Terrible quality', 'Good value for money',
    'Battery dies too fast', 'Perfect for daily use', 'Bluetooth keeps disconnecting',
    'Comfortable fit', 'Sound quality is amazing', 'Broke after 2 weeks',
    'Best purchase this year', 'Not worth the price', 'Excellent noise cancellation'
] * (n // 12 + 1)
bodies = [
    'I have been using this for a month and it works perfectly. The sound quality is excellent and battery lasts all day.',
    'The bluetooth connection drops every 5 minutes. Very frustrating. Had to return it.',
    'Comfortable to wear for long periods. Good noise cancellation. Battery could be better though.',
    'Stopped working after 2 weeks. The charging case lid broke. Poor build quality.',
    'Amazing sound for the price. Bass is deep and clear. Highly recommend.',
    'The earbuds keep falling out of my ears. Tried all the different tips but none fit well.',
    'Battery only lasts 3 hours, not the 8 hours advertised. Disappointed.',
    'Perfect for gym workouts. Stays in place and sweat resistant. Love the design.',
] * (n // 8 + 1)

df = pd.DataFrame({
    'rating': ratings,
    'title': [titles[i % len(titles)] for i in range(n)],
    'body': [bodies[i % len(bodies)] for i in range(n)],
    'date': pd.date_range('2025-01-01', periods=n, freq='D'),
    'verified': np.random.choice([True, False], n, p=[0.8, 0.2]),
    'helpful_votes': np.random.poisson(2, n)
})

print(f'Loaded {len(df)} reviews')
print(f'Rating distribution:\n{df["rating"].value_counts().sort_index()}')
df.head()

## 3. Data Preprocessing

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_title'] = df['title'].apply(clean_text)
df['clean_body'] = df['body'].apply(clean_text)
df['full_text'] = df['clean_title'] + '. ' + df['clean_body']
df = df[df['full_text'].str.len() > 10]

df['sentiment_label'] = df['rating'].map({
    1: 'negative', 2: 'negative', 3: 'neutral', 4: 'positive', 5: 'positive'
})

print(f'After cleaning: {len(df)} reviews')
print(f'Sentiment distribution:\n{df["sentiment_label"].value_counts()}')

## 4. VADER Quick Sentiment Analysis

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def vader_sentiment(text):
    scores = analyzer.polarity_scores(text)
    if scores['compound'] >= 0.05:
        return 'positive', scores['compound']
    elif scores['compound'] <= -0.05:
        return 'negative', scores['compound']
    else:
        return 'neutral', scores['compound']

results = df['full_text'].apply(vader_sentiment)
df['vader_label'] = results.apply(lambda x: x[0])
df['vader_score'] = results.apply(lambda x: x[1])

print('VADER Sentiment Distribution:')
print(df['vader_label'].value_counts())
print(f'\nAccuracy vs rating-based label: {(df["vader_label"] == df["sentiment_label"]).mean():.1%}')

## 5. BERTopic Topic Modeling

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

topic_model = BERTopic(
    embedding_model=embedding_model,
    nr_topics='auto',
    min_topic_size=5,
    language='english',
    verbose=True
)

topics, probs = topic_model.fit_transform(df['full_text'].tolist())
df['topic'] = topics

print('\n=== Topic Overview ===')
topic_info = topic_model.get_topic_info()
topic_info.head(15)

## 6. Negative Review Topic Analysis

In [ ]:
neg_df = df[df['rating'] <= 2]
print(f'Total negative reviews: {len(neg_df)}')
print(f'\n=== Top Negative Review Topics ===')

neg_topics = neg_df.groupby('topic').size().sort_values(ascending=False)
for topic_id in neg_topics.head(8).index:
    if topic_id == -1:
        continue
    keywords = topic_model.get_topic(topic_id)
    keyword_str = ', '.join([w for w, _ in keywords[:5]])
    count = neg_topics[topic_id]
    print(f'\nTopic {topic_id} ({count} negative reviews): {keyword_str}')
    examples = neg_df[neg_df['topic'] == topic_id]['full_text'].head(2)
    for ex in examples:
        print(f'  → {ex[:120]}...')

## 7. Visualization

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Rating distribution
fig = px.histogram(df, x='rating', color='vader_label',
                   title='Rating Distribution with Sentiment',
                   color_discrete_map={'positive': 'green', 'negative': 'red', 'neutral': 'gray'})
fig.show()

# Word cloud for negative reviews
neg_text = ' '.join(neg_df['full_text'].tolist())
wc = WordCloud(width=800, height=400, background_color='white', max_words=80, colormap='Reds').generate(neg_text)
plt.figure(figsize=(12, 6))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Negative Review Word Cloud', fontsize=16)
plt.show()

# Monthly rating trend
df['month'] = pd.to_datetime(df['date']).dt.to_period('M').astype(str)
monthly = df.groupby('month').agg({'rating': 'mean', 'full_text': 'count'}).reset_index()
monthly.columns = ['Month', 'Avg Rating', 'Review Count']
fig = go.Figure()
fig.add_trace(go.Bar(x=monthly['Month'], y=monthly['Review Count'], name='Count'))
fig.add_trace(go.Scatter(x=monthly['Month'], y=monthly['Avg Rating'], name='Avg Rating', yaxis='y2'))
fig.update_layout(title='Monthly Review Trend', yaxis2=dict(overlaying='y', side='right', range=[1,5]))
fig.show()

## 8. Export Results

In [ ]:
# Export analyzed data
df.to_csv('reviews_analyzed.csv', index=False)
topic_info.to_csv('topics_summary.csv', index=False)

print('Exported: reviews_analyzed.csv, topics_summary.csv')
print(f'\n=== Analysis Summary ===')
print(f'Total reviews: {len(df)}')
print(f'Average rating: {df["rating"].mean():.2f}')
print(f'Negative rate: {(df["rating"] <= 2).mean():.1%}')
print(f'Topics found: {len(topic_info) - 1}')
print(f'Top negative topic: {neg_topics.index[0] if len(neg_topics) > 0 else "N/A"}')